# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR² colorectal cancer survivors dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library.

### Dataset Source

The dataset Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset object
dataset = mlc.Dataset(croissant_url)

# Metadata as an object
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview

List all available record sets, their `@id` and fields. Each record set and field are referenced by their unique `@id` (IRI).

In [ ]:
# Get all available record set ids
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"  RecordSet name: {rs.name}\n    @id: {rs.id}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

Let's display the first few records in each record set using the `@id` notation.


In [ ]:
for rs in dataset.record_sets:
    print(f"\nRecordSet {rs.name} (@id: {rs.id}) sample rows:")
    records = list(dataset.records(record_set=rs.id))
    if records:
        for rec in records[:2]:
            print(rec)
    else:
        print("  (No records loaded)")

## 3. Data Extraction

Load the primary record set(s) (by `@id`) into DataFrames for analysis.

We first collect all record set `@id`s and display the columns for the main clinical record set.

In [ ]:
# List all record set ids for reference
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record sets @ids:")
for idx, rid in enumerate(record_set_ids):
    print(f"  [{idx}] {rid}")

# In this dataset, the main table appears to be the single tabular record set. We'll use the first one.
main_record_set_id = record_set_ids[0]  # If multiple, select accordingly

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
    else:
        dataframes[rs_id] = pd.DataFrame()

# Show available columns in the main record set
print(f"Columns in primary record set ({main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

As an example, perform basic operations on a numeric field and group by a key attribute. All fields referenced by their Croissant `@id`.

In [ ]:
# Identify a numeric field.
# Let's use 'Age_at_Second_Primary_Diagnosis' if present, as a hypothetical example (replace with actual @id from schema above).
# For a real notebook, check the actual column names, e.g., 'http://senscience.ai/fields/Age_at_Second_Primary_Diagnosis'

df = dataframes[main_record_set_id]

# Print columns for human inspection to pick fields by @id
print('Available columns:', df.columns.tolist())

# Find a numeric field (try common patterns)
# If unknown, let's pick the first float/int type column
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    # Try to coerce (in case data loaded as objects)
    for col in df.columns:
        try:
            pd.to_numeric(df[col])
            numeric_field_id = col
            break
        except Exception:
            continue
if not numeric_field_id:
    raise ValueError("Couldn't identify a numeric field in the dataset.")

print(f"Selected numeric field for EDA: {numeric_field_id}")

threshold = 50
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (showing up to 5):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Pick a group-by field: use the first non-numeric column as an example (typically a categorical field, such as 'Sex' or diagnosis site)
group_field = None
for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        group_field = col
        break
if group_field:
    print(f"\nGrouping by field (Croissant @id): {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(grouped_df.head())
else:
    print("Could not find a categorical field for grouping.")

## 5. Visualization

Plot distribution of the selected numeric field and compare across groups. Plot references field `@id`s in the axis labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (@id)")
plt.ylabel("Count")
plt.show()

if group_field is not None:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(f"{group_field} (@id)")
    plt.ylabel(f"{numeric_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

We demonstrated how to load, inspect, and process the FAIR² colorectal cancer survivors dataset using the [mlcroissant](https://github.com/mlcommons/croissant-python) library referencing all elements by Croissant `@id`.

- Metadata and structure can be explored programmatically.
- All entities are referenced using stable `@id` fields ensuring clear, reproducible workflows.
- The loaded DataFrame enables further clinical or statistical analysis using Pandas, NumPy, and Python visualization libraries.

Consult the [mlcroissant documentation](https://mlcommons.github.io/croissant-python/) to learn how to chain further transformations or extract subsets by field/record set.